# Interference screen: rank candidate TARGETs by the impact of their inhibition on a CORE network

**Inputs: `CORE` (receptor / disease-causal gene) and `TARGETS` (a list of candidate genes to inhibit).**

The CORE network (receptor → intermediates → TFs → DoRothEA genes) is computed once. For every candidate the pathway-interference model of
`pathway_interference.ipynb` is run: a "TARGET inhibited" perturbation enters the CORE network through the candidate's own reach and propagates
along CORE's signed edges. Each candidate is scored on:

| column | meaning |
|---|---|
| `flow_change_index` | fraction of CORE's TFs whose activity changes |
| `tf_delta_total` | Σ \|delta\| over CORE's TFs — magnitude of the transcriptional change (relative units) |
| `direct_entries` | CORE nodes that receive a direct edge from the candidate's side — the mechanistic footprint (residues in the per-target tables) |
| `down_share` | share of changed TFs that go DOWN: 1 = inhibition silences CORE's output, 0 = it releases brakes (output rises) |
| `baseline_z` | z of `tf_delta_total` vs degree-matched random genes against the same CORE |
| `composite` | mean percentile rank of index, \|delta\| and direct entries (baseline_z breaks ties) — the ranking |

Difference from `crosstalk_screen`: that one scores *proximity and overlap* of two walks; this one scores the *predicted consequence* of inhibiting the candidate for the CORE network's output. Use both; they usually agree on the top, and disagreements are informative (a close interactor with no consequence, or a distant one with a large one).

## 1. Parameters

In [ ]:
CORE = "FGFR3"
TARGETS = """
HSP90AA1
MAPK1
AKT1
MTOR
EGFR
GSK3B
"""                    # plain text / list / path to a file (first column)
DEMO = False
CONFIG = None
TISSUE = None
MAX_DEPTH = MIN_SCORE = HUB_PENALTY = None   # None = YAML walk: section
N_RANDOM = 20          # degree-matched random genes for the baseline (0 = skip)
TARGET_NET_DIR = ".."

## 2. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(TARGET_NET_DIR))
from targetnet.api import direction_figure, screen_interference, interference_screen_figure, necessity_figure, research_pathway_interference, summarize_pathway_interference, figure
from targetnet.plots import use_cdn_renderer
use_cdn_renderer()
import pandas as pd
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 180)
OUT_ROOT = os.path.join(TARGET_NET_DIR, "runs")

## 3. Run the screen

In [ ]:
s = screen_interference(CORE, TARGETS, config=CONFIG, demo=DEMO, tissue=TISSUE, max_depth=MAX_DEPTH, min_score=MIN_SCORE,
                        hub_penalty=HUB_PENALTY, n_random=N_RANDOM, out=OUT_ROOT)
print("outputs:", s["run_dir"])

## 4. Ranking

In [ ]:
tab = s["table"]
tab[["rank","target","tfs_changed","tfs_down","tfs_up","flow_change_index","tf_delta_total","tf_delta_down","tf_delta_up","net_direction","direct_entries","genes_down","genes_up","baseline_z","composite","top_tfs"]]

## 5. Screen view
x = fraction of CORE TFs changed, y = total |delta| at the TFs, size = direct entry points, colour = direction (blue = inhibition mostly silences CORE output, red = mostly releases brakes).

In [ ]:
interference_screen_figure(s).show()

### Direction of the effect per candidate
Diverging bars: blue to the left = the part that goes **down** / opposes, red to the right = the part that goes **up** / agrees. A candidate with both bars long is context-dependent (feedback), not neutral.

In [ ]:
direction_figure(s).show()

### Bottleneck or parallel input? (necessity view)

`flow_change_index` is the **linear amplitude**: how much of the CORE-driven flow is removed when the candidate is inhibited. A receptor feeding the same trunk in parallel (EGFR on the FGFR3 → RAS/PI3K trunk) scores high here, yet an EGFR inhibitor does not block the FGFR route. The **necessity index** answers that second question: CORE is walked again with the candidate removed; a CORE TF counts as *lost* when no route remains, *weakened* when only a much poorer detour remains (`interference.weak_ratio`). `class` combines both axes with the direction: *bottleneck* (blocks CORE output), *parallel input* (amplitude down, route intact = redundant), *brake*, *minor*, *none*. Per-candidate details: `route_loss_<gene>.csv`.

In [ ]:
# quadrant view: x = necessity (route loss), y = amplitude (flow-change index); size = Σ|relative change|, colour = net direction
necessity_figure(s).show()
s["table"][["rank", "target", "class", "necessity_index", "tfs_lost", "tfs_weakened", "flow_change_index", "tf_delta_total", "net_direction", "composite"]]

### The scores on the CORE map
The CORE network as a cell cross-section with every candidate as a diamond: size = Σ|delta| at the CORE TFs when inhibited, colour = net direction (blue silences, red raises, grey balanced), label = rank. Candidates outside the network stand in the left margin at the height of their strongest entry node; thin lines point to their top entry nodes.

In [ ]:
figure(s["views"]["core_map"]).show()

## 6. What each candidate changes
Per-target tables of the changed CORE nodes (delta, direct / propagated, via). Change `HIT`.

In [ ]:
HIT = tab.target.iloc[0] if len(tab) else None
s["per_target"].get(HIT, pd.DataFrame()).head(25) if HIT else None

## 7. Full view of one hit
The cell cross-section with the predicted change, and the entry edges with residues.

In [ ]:
if HIT:
    p = research_pathway_interference(CORE, HIT, config=CONFIG, demo=DEMO, tissue=TISSUE, out=OUT_ROOT)
    print(summarize_pathway_interference(p))
    figure(p["views"]["cell_flow_inhibition"]).show()

## 8. How to interpret
1. **`flow_change_index` and `tf_delta_total` together.** A high index with a small total = many TFs nudged weakly (a diffuse, probably indirect effect); a low index with a large total = a few TFs hit hard (a focused mechanism) — the second is the better drug hypothesis.
2. **`down_share` decides the pharmacology.** Near 1: inhibiting the candidate switches CORE's transcriptional output off — the intended effect if CORE drives disease. Near 0: inhibition *raises* CORE output (feedback release) — a liability, or an opportunity if CORE is protective. Around 0.5: context-dependent; read the per-target table.
3. **`direct_entries` with residues** (per-target tables, `entry` = direct) are the testable claims: CORE stimulation ± candidate inhibitor, phosphoproteomics on those sites.
4. **`baseline_z`** is the only absolute statement: z > 2 means the candidate changes the CORE network more than genes of similar connectivity do.
5. **Skipped candidates** had no walk or no entry at these thresholds — a database gap or a truly unrelated gene; `signor-lookup --gene X` tells which.
6. Magnitudes are relative within the run; compare candidates, not runs.